# Purdue SNN Point Cloud — Full Experiment Pipeline (Kaggle Version)

Runs all experiments from `run_all_experiments.py` on Kaggle with GPU.

**Steps:**
1. GPU check
2. Prepare Kaggle workspace (copy code from /kaggle/input)
3. Apply bug patches
4. Install dependencies
5. Download datasets (ModelNet10, ModelNet40, ScanObjectNN)
6. Smoke test
7. Quick run (ModelNet10, 50 epochs)
8. Full run (MN10 + MN40 + ScanObjectNN, 150 epochs)
9. View results

> **Before running:** Accelerator → GPU T4 x2 · Internet → ON

## 1 · GPU Check

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
else:
    print('WARNING: No GPU — go to Settings → Accelerator → GPU T4 x2')

## 2 · Kaggle Workspace Preparation

Add your `purdueprj` folder as a Kaggle Dataset input:
1. Zip your local `purdueprj/` folder → `purdueprj.zip`
2. Kaggle notebook → **Add Data** → **Upload** → upload the zip
3. It will appear as `/kaggle/input/purdueprj/` (or similar name)

In [ ]:
import os, sys, shutil

LOCAL_PROJECT = '/kaggle/working'
os.makedirs(LOCAL_PROJECT, exist_ok=True)
os.chdir(LOCAL_PROJECT)
print('Working dir:', os.getcwd())

if not os.path.exists('run_all_experiments.py'):
    print('\nSearching for purdueprj codebase in /kaggle/input/ ...')
    purdue_code_dir = None

    print('Available inputs:')
    for item in os.listdir('/kaggle/input'):
        print(f'  - {item}')

    for root, dirs, files in os.walk('/kaggle/input'):
        if 'run_all_experiments.py' in files:
            purdue_code_dir = root
            break

    if purdue_code_dir:
        print(f'\nFound codebase at {purdue_code_dir}')
        print('Copying to /kaggle/working/ ...')
        for item in os.listdir(purdue_code_dir):
            if item.startswith('.') or item in ('results', '__pycache__'):
                continue
            s = os.path.join(purdue_code_dir, item)
            d = os.path.join(LOCAL_PROJECT, item)
            if os.path.isdir(s):
                if os.path.exists(d):
                    shutil.rmtree(d)
                shutil.copytree(s, d)
            else:
                shutil.copy2(s, d)
        print('Copy complete.')
    else:
        raise FileNotFoundError(
            'Could not find run_all_experiments.py in /kaggle/input/\n'
            'Add your purdueprj folder as a dataset input first.'
        )

if os.path.exists('run_all_experiments.py'):
    print('Codebase ready in /kaggle/working')
    !ls -F
else:
    print('ERROR: Codebase not found.')

## 3 · Apply Bug Patches

Four fixes applied in-place after the workspace copy:
- **e3dsnn retain_graph** — detach membrane state (TBPTT) in `E3DBlock.forward`
- **E3DSNNModel mem detach** — detach top-level `self.mem` in `forward_step`
- **Energy formula** — remove spurious `× num_slices` factor (T cancels)
- **ScanObjectNN unpack** — `eval_model` returns 5 values, was unpacked as 4

In [ ]:
import re

def patch_file(path, old, new, label):
    with open(path) as f:
        src = f.read()
    if old not in src:
        print(f'  [SKIP] {label} — pattern not found (already patched?)')
        return
    src = src.replace(old, new, 1)
    with open(path, 'w') as f:
        f.write(src)
    print(f'  [OK]   {label}')

# Patch 1a: e3dsnn_backbone.py — detach mem1
patch_file(
    'models/e3dsnn_backbone.py',
    'spk1, self.mem1 = self._ilif(self.mem1, out)',
    'spk1, self.mem1 = self._ilif(self.mem1.detach(), out)',
    'e3dsnn: detach mem1 (TBPTT)'
)

# Patch 1b: e3dsnn_backbone.py — detach mem2
patch_file(
    'models/e3dsnn_backbone.py',
    'spk2, self.mem2 = self._ilif(self.mem2, out2)',
    'spk2, self.mem2 = self._ilif(self.mem2.detach(), out2)',
    'e3dsnn: detach mem2 (TBPTT)'
)

# Patch 2: model_zoo.py — detach top-level mem in E3DSNNModel
patch_file(
    'models/model_zoo.py',
    'self.mem = self.tau * self.mem + feat',
    'self.mem = self.tau * self.mem.detach() + feat',
    'E3DSNNModel: detach top-level mem (TBPTT)'
)

# Patch 3: run_all_experiments.py — energy formula (remove x T)
patch_file(
    'run_all_experiments.py',
    'efficiency = (e_ac / e_mac) * mean_fr * args.num_slices',
    'efficiency = (e_ac / e_mac) * mean_fr',
    'energy formula: remove spurious x num_slices'
)

# Patch 4: run_all_experiments.py — scanobjectnn unpack 4->5
patch_file(
    'run_all_experiments.py',
    'val_acc, mean_exit, _, _ = eval_model(',
    'val_acc, mean_exit, _, _, _ = eval_model(',
    'scanobjectnn: eval_model returns 5 values (was unpacked as 4)'
)

print('\nAll patches done.')

## 4 · Install Dependencies

In [ ]:
!pip install -q kagglehub trimesh h5py numpy matplotlib pandas scikit-learn tqdm

## 5 · Download Datasets

**ScanObjectNN note:** If you see 403 errors, you need to accept the dataset terms:
1. Go to https://www.kaggle.com/datasets/mahmoud88/scanobjectnn
2. Click **Download** and accept the rules
3. Re-run this cell

If ScanObjectNN fails entirely, `SONN_DIR` will be `None` and the `scanobjectnn` group is automatically skipped.

In [ ]:
import kagglehub, os

# ── ModelNet10 ────────────────────────────────────────────────────────────────
MN10_DIR = None
print('Downloading ModelNet10 (~200 MB) ...')
try:
    p = kagglehub.dataset_download('balraj98/modelnet10-princeton-3d-object-dataset')
    for root, dirs, _ in os.walk(p):
        if 'ModelNet10' in dirs:
            MN10_DIR = os.path.join(root, 'ModelNet10')
            break
    if MN10_DIR is None:
        MN10_DIR = p
    print(f'  ModelNet10: {MN10_DIR}')
except Exception as e:
    print(f'  ModelNet10 FAILED: {e}')

# ── ModelNet40 ────────────────────────────────────────────────────────────────
MN40_DIR = None
print('Downloading ModelNet40 (~1.9 GB) ...')
try:
    p = kagglehub.dataset_download('balraj98/modelnet40-princeton-3d-object-dataset')
    for root, dirs, _ in os.walk(p):
        if 'ModelNet40' in dirs:
            MN40_DIR = os.path.join(root, 'ModelNet40')
            break
    if MN40_DIR is None:
        MN40_DIR = p
    print(f'  ModelNet40: {MN40_DIR}')
except Exception as e:
    print(f'  ModelNet40 FAILED: {e}')

# ── ScanObjectNN ──────────────────────────────────────────────────────────────
# If you see 403, visit https://www.kaggle.com/datasets/mahmoud88/scanobjectnn
# and accept the dataset rules, then re-run.
SONN_DIR = None
print('Downloading ScanObjectNN (~1.5 GB) ...')
for slug in ['mahmoud88/scanobjectnn', 'kevinmayer/scanobjectnn',
             'ranjeetjain3/scanobjectnn', 'fepegar/scanobjectnn']:
    try:
        p = kagglehub.dataset_download(slug)
        if os.path.exists(os.path.join(p, 'h5_files')):
            SONN_DIR = os.path.join(p, 'h5_files')
        else:
            SONN_DIR = p
        print(f'  ScanObjectNN ({slug}): {SONN_DIR}')
        break
    except Exception as e:
        print(f'  {slug} failed: {e}')

if SONN_DIR is None:
    print('  Fallback: wget from official source ...')
    !wget -q -O /tmp/sonn.zip http://hkust-vgd.ust.hk/scanobjectnn/h5_files.zip
    if os.path.exists('/tmp/sonn.zip') and os.path.getsize('/tmp/sonn.zip') > 1_000_000:
        !unzip -q /tmp/sonn.zip -d /tmp/ScanObjectNN
        !rm /tmp/sonn.zip
        SONN_DIR = '/tmp/ScanObjectNN/h5_files'
        print(f'  ScanObjectNN (wget): {SONN_DIR}')
    else:
        print('  wget also failed. ScanObjectNN will be skipped.')
        !rm -f /tmp/sonn.zip

print(f'\nMN10  = {MN10_DIR}')
print(f'MN40  = {MN40_DIR}')
print(f'SONN  = {SONN_DIR}')
if SONN_DIR is None:
    print('\nNOTE: ScanObjectNN unavailable — scanobjectnn group will be skipped in runs.')

## 6 · Smoke Test (~2 min, random data)

In [ ]:
!python run_all_experiments.py \
    --smoke_test \
    --groups comparison scaling slicing conversion early_exit \
    --datasets modelnet10 modelnet40 \
    --out_dir results/smoke

## 7 · Quick Run — ModelNet10 (50 epochs, ~8 h on T4)

Runs all core groups on ModelNet10. Good for validating results before the full run.

In [ ]:
import subprocess, sys

cmd = [
    sys.executable, 'run_all_experiments.py',
    '--mn10_root', MN10_DIR or '',
    '--datasets', 'modelnet10',
    '--groups', 'comparison', 'scaling', 'slicing', 'conversion', 'early_exit',
    '--epochs', '50',
    '--batch_size', '16',
    '--out_dir', 'results/mn10_quick',
]
print('CMD:', ' '.join(cmd))
result = subprocess.run(cmd)
print('Exit code:', result.returncode)

## 8 · Full Run — MN10 + MN40 + ScanObjectNN (150 epochs, ~48 h)

`scanobjectnn` group is automatically included only when `SONN_DIR` is not None.

In [ ]:
import subprocess, sys

base_groups = ['comparison', 'scaling', 'slicing', 'conversion',
               'early_exit', 'multi_seed', 't_sweep']
if SONN_DIR:
    base_groups.append('scanobjectnn')
    print('ScanObjectNN available — including scanobjectnn group.')
else:
    print('ScanObjectNN not available — scanobjectnn group skipped.')

datasets = ['modelnet10', 'modelnet40']

cmd = [
    sys.executable, 'run_all_experiments.py',
    '--mn10_root', MN10_DIR or '',
    '--mn40_root', MN40_DIR or '',
    '--datasets', *datasets,
    '--groups', *base_groups,
    '--epochs', '150',
    '--batch_size', '16',
    '--out_dir', 'results/full',
]
if SONN_DIR:
    cmd += ['--sonn_root', SONN_DIR]

print('CMD:', ' '.join(cmd))
result = subprocess.run(cmd)
print('Exit code:', result.returncode)

## 9 · View Results

Results are in the **Output** pane on the right. Download via the three-dot menu.

In [ ]:
import pathlib
from IPython.display import display

# Use whichever results dir exists
for candidate in ('results/full', 'results/mn10_quick', 'results/smoke'):
    results_dir = pathlib.Path(candidate)
    if results_dir.exists():
        break

table_path = results_dir / 'comparison_table.txt'
if table_path.exists():
    print(table_path.read_text())
else:
    print('No comparison_table.txt yet — check run completed.')

for p in sorted(results_dir.rglob('*.png')):
    print(f'\n── {p.name} ──')
    from IPython.display import Image
    display(Image(str(p)))